# VectrixDB Document Index

The Document Index provides hierarchical document management with automatic chunking.

**No embedding model required!** Document Index handles:
- Document parsing (markdown, PDF, text)
- Tree structure (Documents > Sections > Chunks)
- Smart chunking with overlap

You can then embed the chunks separately if needed for vector search.

## Basic Setup

In [ ]:
from vectrixdb import VectrixDB
from vectrixdb.core.document_index import DocumentIndex

# Create database
db = VectrixDB(path="doc_index_demo")

# Create document index (uses storage backend, no embedding needed)
doc_index = DocumentIndex(db._storage)

print("Document Index created - no embedding model needed!")

## Index a Markdown Document

In [ ]:
markdown_content = """
# VectrixDB Guide

VectrixDB is a powerful vector database for AI applications.

## Installation

Install VectrixDB using pip:
```bash
pip install vectrixdb
```

## Quick Start

Create a database and add vectors:
```python
from vectrixdb import VectrixDB
db = VectrixDB("my_db")
```

### Adding Vectors

You can add vectors with metadata for filtering.

### Searching

Search returns the most similar vectors.

## Storage Backends

VectrixDB supports multiple storage backends including SQLite, 
PostgreSQL, CosmosDB, Delta Lake, and Lakebase.
"""

# Index the document
doc_info = doc_index.index_text(
    doc_id="guide_001",
    content=markdown_content,
    title="VectrixDB Guide",
    doc_type="markdown",
    metadata={"version": "1.0", "author": "VectrixDB Team"}
)

print(f"Indexed: {doc_info.title}")
print(f"  Sections: {doc_info.section_count}")
print(f"  Nodes: {doc_info.node_count}")

## View Document Tree Structure

In [ ]:
# Get all nodes (sections) in the document
nodes = doc_index.get_document_nodes("guide_001")

print("Document Tree:")
for node in nodes:
    indent = "  " * (node.level - 1)
    print(f"{indent}[Level {node.level}] {node.title}")
    if node.summary:
        print(f"{indent}  Summary: {node.summary[:60]}...")

## Get Chunks for Embedding

Get text chunks ready for embedding (if you want vector search).

In [ ]:
# Get chunks with customizable size and overlap
chunks = doc_index.get_chunks(
    doc_id="guide_001",
    chunk_size=500,
    chunk_overlap=50
)

print(f"Generated {len(chunks)} chunks:")
for i, chunk in enumerate(chunks[:3], 1):
    print(f"\nChunk {i}:")
    print(f"  ID: {chunk.chunk_id}")
    print(f"  Heading: {chunk.heading}")
    print(f"  Text: {chunk.text[:80]}...")

## Navigate Sections

In [ ]:
# Get a specific section by title
section = doc_index.get_section("guide_001", "Installation")

if section:
    print(f"Section: {section.title}")
    print(f"Level: {section.level}")
    print(f"Content: {section.text[:200]}...")

# Get section path (breadcrumb)
path = doc_index.get_section_path(section.node_id)
print(f"\nPath: {' > '.join(path)}")

## Get Child Sections

In [ ]:
# Find Quick Start section and get its children
quick_start = doc_index.get_section("guide_001", "Quick Start")

if quick_start:
    children = doc_index.get_children(quick_start.node_id)
    print(f"'{quick_start.title}' has {len(children)} subsections:")
    for child in children:
        print(f"  - {child.title}")

## Index Plain Text

In [ ]:
plain_text = """
VectrixDB is designed for modern AI workloads.

It provides fast similarity search using HNSW indexing.

The database supports hybrid search combining dense and sparse vectors.

Integration with Databricks enables enterprise governance.
"""

doc_info = doc_index.index_text(
    doc_id="notes_001",
    content=plain_text,
    title="VectrixDB Notes",
    doc_type="text"
)

print(f"Indexed plain text: {doc_info.node_count} paragraphs")

## Index a File

In [ ]:
# Index a file (auto-detects type from extension)
# doc_info = doc_index.index_file(
#     file_path="/path/to/document.md",
#     doc_id="doc_from_file"
# )

print("File indexing: auto-detects .md, .txt, .pdf extensions")

## List All Documents

In [ ]:
documents = doc_index.list_documents()

print(f"Total documents: {len(documents)}")
for doc in documents:
    print(f"  - {doc.doc_id}: {doc.title} ({doc.doc_type.value})")

## Get Statistics

In [ ]:
stats = doc_index.get_stats()

print("Index Statistics:")
print(f"  Total documents: {stats['total_documents']}")
print(f"  Total nodes: {stats['total_nodes']}")
print(f"  Total sections: {stats['total_sections']}")
print(f"  By type: {stats['by_type']}")

## Using Chunks with Vector Search (Optional)

If you want vector search, embed chunks and add to a collection:

In [ ]:
# Example workflow with embedding (requires embedding model)

# 1. Get chunks
# chunks = doc_index.get_chunks("guide_001")

# 2. Embed chunk texts (use your embedding model)
# texts = [chunk.text for chunk in chunks]
# vectors = your_embedding_model.encode(texts)

# 3. Add to VectrixDB collection
# coll = db.create_collection("docs", dimension=384)
# coll.add(
#     ids=[c.chunk_id for c in chunks],
#     vectors=vectors,
#     metadata=[{"doc_id": c.doc_id, "heading": c.heading} for c in chunks]
# )

print("Document Index + embedding = powerful document search!")

## Delete Document

In [ ]:
# Delete document and all its nodes
doc_index.delete_document("notes_001")

print(f"Remaining documents: {len(doc_index.list_documents())}")

## Cleanup

In [ ]:
import shutil
import os

db.close()

if os.path.exists("doc_index_demo"):
    shutil.rmtree("doc_index_demo")
    print("Cleaned up demo folder")